In [1]:
%cd ..

c:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [7]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [8]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [49]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\cost-2025\chi_phi_2025"
# resource_name = "costs"
resource_name = "cost_items"

df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "CP"

df.head()
print(len(df))
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

300
✅ Done: cost_items.json created


In [50]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  cost_items
cost_items
Replace Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/cost_items/data_cost_items_20260318_205150.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [51]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T01_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="DT",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "DT"
df["source_file"] = "doanh_thu_chi_phi_cong_ty_T01_2026.xlsx"

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


6
✅ Done: cost_items.json created


In [52]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/cost_items/data_cost_items_20260318_205155.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [54]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T01_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="CP",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "CP"
df["source_file"] = filename.split("\\")[-1]

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


25
✅ Done: cost_items.json created


In [55]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/cost_items/data_cost_items_20260318_205215.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [56]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T02_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="CP",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "CP"
df["source_file"] = filename.split("\\")[-1]

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


25
✅ Done: cost_items.json created


In [57]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/cost_items/data_cost_items_20260318_205227.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [58]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T02_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="DT",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "DT"
df["source_file"] = filename.split("\\")[-1]

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


6
✅ Done: cost_items.json created


In [59]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/cost_items/data_cost_items_20260318_205242.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [9]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T03_2026.xlsx"
print(filename)
df = read_excel_and_normalize_columns(
      filename,
    sheet_name="DT",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "DT"
df["source_file"] = filename.split("\\")[-1]

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T03_2026.xlsx
6
✅ Done: cost_items.json created


In [10]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/finance_raw/cost_items/data_cost_items_20260407_162613.parquet
bucket=vcs-raw , key=finance-raw/cost_items/data_cost_items_20260407_162613.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False

In [11]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "Data type": "data_type",
    "Khoản mục cấp 2": "expense_category_level_2",
    "Mã KM phí": "expense_code",
    "Triệu đồng": "amount_million_vnd",
    
    "Mã KM":"expense_code",
    "KM phí chi tiết": "expense_category_level_2"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\doanh_thu_chi_phi_cong_ty_T03_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="CP",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["data_type"] = "CP"
df["source_file"] = filename.split("\\")[-1]

# resource_name = "costs"
resource_name = "cost_items"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


25
✅ Done: cost_items.json created


In [12]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  cost_items
cost_items
Add Upload  s3a://vcs-raw/finance-raw/cost_items ./tmp/data/finance_raw/cost_items/data_cost_items_20260407_162619.parquet
bucket=vcs-raw , key=finance-raw/cost_items/data_cost_items_20260407_162619.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/cost_items
Uploaded SQL definition


False